# DenseNet201 chest X-ray notebook

Author: **Aaditya Shah**.

This notebook trains a frozen ImageNet DenseNet201 on NIH ChestX-ray14. The Mass and Nodule tables in the README come from the five-backbone training code in `src`, run through `notebooks/01_train.ipynb`.


Train a multi-label classifier on NIH ChestX-ray14. Each image can carry any of the 14 pathology labels plus "No Finding". The labels of interest for this project are Mass and Nodule. Dataset: NIH Chest X-ray dataset.


In [ ]:
import numpy as np 
import pandas as pd
import os
import glob
import itertools
import time

import seaborn as sns

import tensorflow as tf
import tensorflow_addons as tfa
import keras

import matplotlib.pyplot as plt

from tensorflow.keras import backend as K

from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LayerNormalization, Input, LSTM, GRU, TimeDistributed
from tensorflow.keras.layers import Conv2D, BatchNormalization, ReLU, MaxPooling2D, Flatten, Dropout, GlobalAveragePooling2D, Dense, Softmax, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam

from pandas import DataFrame
from typing import List
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
import tensorflow as tf

device_name = tf.test.gpu_device_name()

if "GPU" not in device_name:
    print("GPU device not found")
    
print('Found GPU at: {}'.format(device_name))

print("GPU", "available (YESS!!!!)" if tf.config.list_physical_devices("GPU") else "not available :(")

In [ ]:
# Read the dataframe
import os
from pathlib import Path

DATA_DIR = os.environ.get("DATA_DIR", "data/nih")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "data/output")

df = pd.read_csv(os.path.join(DATA_DIR, "Data_Entry_2017.csv"))

# Rename the columns
df = df.rename(columns={
    "OriginalImagePixelSpacing[x": "OriginalImagePixelSpacingX",
    "y]": "OriginalImagePixelSpacingY",
    "OriginalImage[Width": "OriginalImageWidth",
    "Height]": "OriginalImageHeight"
})

### Format the label by a one hot encoding

# Get all the label
unique_labels = set(itertools.chain.from_iterable(df["Finding Labels"].apply(lambda x : x.split('|')).values)) 

# Initialize a empty matrix 
one_hot_labels = pd.DataFrame(0.0, index=np.arange(len(df)), columns=unique_labels)

# For each row, we get the associated labels and set a 1 to the new corresponding column label 
for index, row in df.iterrows():
    labels = row["Finding Labels"].split('|')
    for label in labels:
        one_hot_labels.iloc[index][label] = 1.0

# Then, we concatane this new dataframe to our original data
df = pd.concat([df, one_hot_labels], axis=1)

df.head()


Next, we are going to create a test set. We will never look at this test set during the training phase.
This test set will allow us to measure the performance of our current model.

In [ ]:
df_train, df_test = train_test_split(df, test_size = 0.33, random_state = 42)
print("Size of training set :", len(df_train), " - testing set:", len(df_test))

In [ ]:
df_train.head()

## Manage Data Leakage

During the split, images from the same patient can land in both sets.
To avoid that leakage, drop test patients who also appear in the train set. The other direction works too. It depends on how large a test set you want.

In [ ]:
patient_train = set(df_train["Patient ID"].values)
patient_test  = set(df_test["Patient ID"].values)

leakage_patient = patient_train.intersection(patient_test)
print(len(leakage_patient))

In [ ]:
# How much this represent based on our set
train_without_leakage = df_train[~df_train["Patient ID"].isin(leakage_patient)]

print("Train")
print(len(train_without_leakage) / len(df_train))
print(len(df_train))
print(len(train_without_leakage))

print()
print("Test")
test_without_leakage = df_test[~df_test["Patient ID"].isin(leakage_patient)]
print(len(test_without_leakage) / len(df_test))
print(len(df_test))
print(len(test_without_leakage))

In [ ]:
# Remvove leakage in the test
df_test = test_without_leakage

## Training phase

For the model, we used the EfficientNetB0 model pretrained.

### Preprocessing

For the preprocessing of our data, we will simply take our gray image and duplicate the channel to obtain 3 channel in total.
Regarding the image size, we will have an image of 224x224.

> Note: For `EfficientNetB0` model, we don't need to normalize our input data. This could be a necessary step for other models.


In [ ]:
NB_LABELS = len(unique_labels)
BATCH_SIZE = 32

X_train, X_valid = train_test_split(df_train, test_size = 0.2, random_state = 42)


In [ ]:
class ChestImagesDataset(tf.keras.utils.Sequence):
    def __init__(self, df, input_folder, batch_size = 32, transform = None, training = True, shuffle = True):
        self.df           = df
        self.input_folder = input_folder
        self.batch_size   = batch_size
        self.transform    = transform
        self.training     = training
        self.shuffle      = shuffle
        
    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac = 1).reset_index(drop = True)
    
    def __len__(self):
        return np.ceil(len(self.df) / self.batch_size).astype(int)
    
    def __getitem__(self, index):
        batch = self.df[index * self.batch_size : (index + 1) * self.batch_size]
        
        images = []
        
        for filename in batch["Image Index"]:
            path = glob.glob(self.input_folder + "*/*/" + filename)[0]
            image = Image.open(path).convert('RGB')
            image = np.array(image)
            
            image = tf.image.resize(image, size=(224,224))
            image = tf.cast(image, dtype = tf.float32)
            
            images.append(image)
        
        images = np.array(images)
        
        if self.training:
            return images, np.array(batch[unique_labels].values)
        else: 
            return images
    
# Load the training data and the validation 
train_generator    = ChestImagesDataset(X_train, DATA_DIR, batch_size = BATCH_SIZE, transform = None, shuffle = True)
valid_generator    = ChestImagesDataset(X_valid, DATA_DIR, batch_size = BATCH_SIZE, transform = None, shuffle = True)

In [ ]:
class MultiLabelCrossEntropy:
    """
    Weight Cross Entropy Loss.
    
    For each class we will found two weights corresponding to the positive and negative frequency of our sample data.
    These weights will manage the way training changes the network. And this has for objectives to manage the unbalanced class issue.
    """
        
    def __init__(self, labels : DataFrame, epsilon = 1e-7):
        
        self.epsilon = epsilon
        
        # Get the size of the data
        self.N = labels.shape[0]
        
        # Get the frequency occurence for each class
        self.freq_pos = np.sum(labels == 1, axis=0) / self.N
        self.freq_neg = np.sum(labels == 0, axis=0) / self.N
        
        # Set the loss weights for each labels 
        self.pos_weights = self.freq_neg
        self.neg_weights = self.freq_pos
        
    def contribution(self):
        """
        Get the weights' contribution for each labels.
        
        Returns :
            - double : Positive contribution
            - double : Negative contribution
        """
        return self.freq_pos * self.pos_weights, self.freq_neg * self.neg_weights
        
        
    def loss(self, y_true, y_pred):
        """
        Return weighted loss value. 
        """
        # Initialize loss to zero
        loss = 0.0
        
        for i in range(len(self.pos_weights)):
            loss += (-1 * K.mean(
                        self.pos_weights[i] * y_true[:,i] * K.log(y_pred[:,i] + self.epsilon))
                    ) + (-1 * K.mean(
                        self.neg_weights[i] * (1 - y_true[:,i]) * K.log(1 - y_pred[:,i] + self.epsilon))
                    )
        return loss
        

In [ ]:
cross_entropy_loss = MultiLabelCrossEntropy(X_train[unique_labels])

pos_contribution, neg_contribution = cross_entropy_loss.contribution()

In [ ]:
labels = list(unique_labels)

data = pd.DataFrame({"Class": labels, "Label": "Positive", "Value": pos_contribution})
data = data.append([{"Class": labels[l], "Label": "Negative", "Value": v} 
                        for l,v in enumerate(neg_contribution)], ignore_index=True)
plt.xticks(rotation=90)
sns.barplot(x="Class", y="Value", hue="Label" ,data=data);

Note : For the model, we could use other alternatives as DenseNet ...

In [ ]:
def create_denseNet_model(cross_entropy_loss):
    
    inputs = Input(shape=(224, 224, 3))
    model = DenseNet201(input_tensor=inputs, weights="imagenet", include_top=False)
    model.trainable = False # Freeze the layer
    
    # Rebuild top
    x = GlobalAveragePooling2D(name="avg_pool")(model.output)
    x = BatchNormalization()(x)

    x = Dropout(0.2, name="top_dropout")(x)
    outputs = Dense(NB_LABELS, activation="sigmoid", name="pred")(x)
    
    # Compile
    model = tf.keras.Model(inputs, outputs, name="DenseNet")
    
    # Note : Possibility to assign a learning rate.
    model.compile(optimizer=Adam(), 
                  loss = cross_entropy_loss.loss,
                  metrics=[
        'accuracy',
        tf.keras.metrics.Recall(),
        tfa.metrics.F1Score(num_classes = NB_LABELS, threshold=0.5),
        tf.keras.metrics.AUC(multi_label = True)
    ])
    
    return model

model = create_denseNet_model(cross_entropy_loss)


In [ ]:
history = model.fit(train_generator, validation_data=valid_generator, epochs = 10, batch_size = BATCH_SIZE)

In [ ]:
# Save the model 
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save(str(Path(OUTPUT_DIR) / "denseNet.keras"))


In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

plt.savefig("model_loss_denseNet")

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

plt.savefig("model_acc_denseNet")

In [ ]:
plt.plot(history.history['recall'])
plt.plot(history.history['val_recall'])

plt.title('model recall')
plt.ylabel('recall')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

plt.savefig("model_recall_denseNet")

In [ ]:
plt.plot(history.history['f1_score'])
plt.plot(history.history['val_f1_score'])

plt.title('model f1_score')
plt.ylabel('f1_score')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

plt.savefig("model_f1_denseNet")

In [ ]:
plt.plot(history.history['auc'])
plt.plot(history.history['val_auc'])

plt.title('model auc')
plt.ylabel('auc')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

plt.savefig("model_auc_denseNet")

## Test phase

In [ ]:
test_generator = ChestImagesDataset(df_test, DATA_DIR, batch_size = BATCH_SIZE, transform = None, training = False, shuffle = False)

predictions = model.predict(test_generator)

In [ ]:
roc_auc_score(df_test[unique_labels].values, predictions)

In [ ]:
## Metrics

def get_roc_curve(labels, predicted_vals, real_vals):
    auc_roc_vals = []
    for i in range(len(labels)):
        try:
            gt = real_vals[:, i]
            pred = predicted_vals[:, i]
            auc_roc = roc_auc_score(gt, pred)
            auc_roc_vals.append(auc_roc)
            fpr_rf, tpr_rf, _ = roc_curve(gt, pred)
            plt.figure(1, figsize=(10, 10))
            plt.plot([0, 1], [0, 1], 'k--')
            plt.plot(fpr_rf, tpr_rf,
                     label=labels[i] + " (" + str(round(auc_roc, 3)) + ")")
            plt.xlabel('False positive rate')
            plt.ylabel('True positive rate')
            plt.title('ROC curve')
            plt.legend(loc='best')
            plt.savefig("roc_curve_denseNet")
        except:
            print(
                f"Error in generating ROC curve for {labels[i]}. "
                f"Dataset lacks enough examples."
            )
    plt.show()
    return auc_roc_vals

print(get_roc_curve(list(unique_labels), predictions, df_test[unique_labels].values))

In [ ]:
from sklearn.metrics import accuracy_score

def get_accuracy_curve(labels, predicted_vals, real_vals):
    accuracy_vals = []
    for i in range(len(labels)):
        try:
            gt = real_vals[:, i]
            pred = predicted_vals[:, i]
            accuracy = accuracy_score(gt, (pred > 0.5).astype(int))
            accuracy_vals.append(accuracy)
        except:
            print(
                f"Error in calculating accuracy for {labels[i]}. "
                f"Dataset lacks enough examples."
            )

    return accuracy_vals

accuracy_vals = get_accuracy_curve(list(unique_labels), predictions, df_test[unique_labels].values)

print(accuracy_vals)

# Plotting accuracy curve
plt.figure(figsize=(20, 5))
plt.plot(labels, accuracy_vals, marker='o', linestyle='-', color='b')
plt.xlabel('Labels')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.show()

plt.savefig("acc_curve_denseNet")

In [ ]:
from sklearn.metrics import f1_score

def get_f1_score_curve(labels, predicted_vals, real_vals):
    f1_score_vals = []
    for i in range(len(labels)):
        try:
            gt = real_vals[:, i]
            pred = predicted_vals[:, i]
            f1 = f1_score(gt, (pred > 0.5).astype(int))
            f1_score_vals.append(f1)
        except:
            print(
                f"Error in calculating F1 score for {labels[i]}. "
                f"Dataset lacks enough examples."
            )

    return f1_score_vals

f1_score_vals = get_f1_score_curve(list(unique_labels), predictions, df_test[unique_labels].values)

print(f1_score_vals)

# Plotting F1 score curve
plt.figure(figsize=(20, 5))
plt.plot(labels, f1_score_vals, marker='o', linestyle='-', color='g')
plt.xlabel('Labels')
plt.ylabel('F1 Score')
plt.title('F1 Score Curve')
plt.show()

plt.savefig("f1_curve_denseNet")

In [ ]:
from sklearn.metrics import recall_score

def get_sensitivity_curve(labels, predicted_vals, real_vals):
    sensitivity_vals = []
    for i in range(len(labels)):
        try:
            gt = real_vals[:, i]
            pred = predicted_vals[:, i]
            sensitivity = recall_score(gt, (pred > 0.5).astype(int))
            sensitivity_vals.append(sensitivity)
        except:
            print(
                f"Error in calculating sensitivity for {labels[i]}. "
                f"Dataset lacks enough examples."
            )

    return sensitivity_vals

sensitivity_vals = get_sensitivity_curve(list(unique_labels), predictions, df_test[unique_labels].values)

print(sensitivity_vals)

# Plotting sensitivity curve
plt.figure(figsize=(23, 5))
plt.plot(labels, sensitivity_vals, marker='o', linestyle='-', color='r')
plt.xlabel('Labels')
plt.ylabel('Sensitivity (Recall)')
plt.title('Sensitivity (Recall) Curve')
plt.show()

plt.savefig("sensitivity_curve_denseNet")

In [ ]:
from sklearn.metrics import confusion_matrix

def get_specificity_curve(labels, predicted_vals, real_vals):
    specificity_vals = []
    for i in range(len(labels)):
        try:
            gt = real_vals[:, i]
            pred = predicted_vals[:, i]
            tn, fp, fn, tp = confusion_matrix(gt, (pred > 0.5).astype(int)).ravel()
            specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
            specificity_vals.append(specificity)
        except:
            print(
                f"Error in calculating specificity for {labels[i]}. "
                f"Dataset lacks enough examples."
            )

    return specificity_vals

specificity_vals = get_specificity_curve(list(unique_labels), predictions, df_test[unique_labels].values)

print(specificity_vals)

# Plotting specificity curve
plt.figure(figsize=(23, 5))
plt.plot(labels, specificity_vals, marker='o', linestyle='-', color='purple')
plt.xlabel('Labels')
plt.ylabel('Specificity')
plt.title('Specificity Curve')
plt.show()

plt.savefig("specificity_curve_denseNet")